# Tests: `fasterai.prune.prune_callback` (source `nbs/prune/prune_callback.ipynb`)

In [ ]:
from fastcore.test import *
from fastcore.foundation import L
from functools import partial
from pathlib import Path
import tempfile
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset
from fastai.callback.core import Callback
from fastai.data.core import DataLoaders
from fastai.learner import Learner
from fastai.optimizer import OptimWrapper
from fasterai.core.criteria import large_final
from fasterai.core.schedule import agp, one_shot
from fasterai.prune.prune_callback import *

In [ ]:
from fastcore.test import *
import warnings

# Construction stores the fraction it was given
cb = PruneCallback(
    pruning_ratio=0.3,
    schedule=agp,
    context='global',
    criteria=large_final
)
test_eq(cb.pruning_ratio, 0.3)
test_eq(cb.context, 'global')
test_eq(cb._is_per_layer, False)

# A percent is read as x/100 for one release, and warns once at construction
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    _pct = PruneCallback(pruning_ratio=30, schedule=agp, context='global', criteria=large_final)
test_eq(_pct.pruning_ratio, 0.3)
test_eq(len(w), 1)
test_eq(w[0].category, FutureWarning)

# Per-layer dict is accepted at construction (regression: dict used to crash in before_fit)
cb_dict = PruneCallback(
    pruning_ratio={'0': 0.3, '3': 0.5},
    schedule=one_shot,
    context='local',
    criteria=large_final
)
test_eq(cb_dict.pruning_ratio, {'0': 0.3, '3': 0.5})
test_eq(cb_dict._is_per_layer, True)

# Validation is idempotent — before_fit re-reads the already-converted fraction
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    _pct._validate_pruning_ratio()
    cb_dict._validate_pruning_ratio()
test_eq(_pct.pruning_ratio, 0.3)
test_eq(cb_dict.pruning_ratio, {'0': 0.3, '3': 0.5})
test_eq(len(w), 0)

# dict + global context is rejected with a clear error
with ExceptionExpected(ValueError, regex='requires'):
    PruneCallback(pruning_ratio={'0': 0.3}, schedule=one_shot, context='global', criteria=large_final)

# Out-of-range, zero and non-numeric ratios are rejected
with ExceptionExpected(ValueError, regex="'0'"):
    PruneCallback(pruning_ratio={'0': 150}, schedule=one_shot, context='local', criteria=large_final)
with ExceptionExpected(ValueError, regex='removes nothing'):
    PruneCallback(pruning_ratio=0, schedule=one_shot, context='local', criteria=large_final)
with ExceptionExpected(TypeError, regex='must be a number'):
    PruneCallback(pruning_ratio='0.3', schedule=one_shot, context='local', criteria=large_final)

In [ ]:
# A prune replaces the parameters of every layer it shrinks: the optimizer must be re-pointed at the
# new ones, or the pruned layers receive no update for the rest of the fit
class _OptProbe(Callback):
    "Watch, around every step, what the optimizer holds and what it remembers"
    order = 60  # after PruneCallback, which prunes in `before_step`
    def __init__(self): self.missed, self.carried, self.channels, self.snapshot = [], [], [], None
    def before_step(self):
        # the output bias survives every prune (only the input side of the head shrinks)
        self.carried.append('grad_avg' in self.learn.opt.state[self.learn.model[-1].bias])
    def after_step(self):
        held = {id(p) for g in self.learn.opt.param_lists for p in g}
        self.missed.append(sum(id(p) not in held for p in self.learn.model.parameters()))
        self.channels.append(self.learn.model[0].out_channels)
        if self.snapshot is None and self.channels[-1] < 16:
            self.snapshot = self.learn.model[0].weight.detach().clone()

def _probe_learner(schedule, probe, **kwargs):
    torch.manual_seed(0)
    model = nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
        nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 10)
    )
    X, y = torch.randn(64, 3, 8, 8), torch.randint(0, 10, (64,))
    dls = DataLoaders.from_dsets(TensorDataset(X[:48], y[:48]), TensorDataset(X[48:], y[48:]),
                                 bs=16, device='cpu')
    cb = PruneCallback(pruning_ratio=0.5, schedule=schedule, context='local', criteria=large_final)
    return Learner(dls, model, loss_func=nn.CrossEntropyLoss(), cbs=[cb, probe], **kwargs)

_probe = _OptProbe()
_learn = _probe_learner(one_shot, _probe)
_learn.fit(2)  # 2 epochs of 3 batches

# (a) the optimizer holds every live parameter after every step — it held 1 of 10 before the rebind
test_eq(_probe.missed, [0] * 6)
# (b) the pruned conv keeps training after the prune — its weights did not move at all before the rebind
test_eq(_probe.channels, [8] * 6)  # one_shot prunes at the first step and never again: same shape throughout
assert _probe.snapshot is not None, "the model was never pruned"
test_eq(_learn.model[0].weight.shape, _probe.snapshot.shape)
assert not torch.equal(_learn.model[0].weight.detach(), _probe.snapshot), \
    "the pruned conv did not change after the prune: the optimizer is stepping dead parameters"
# the optimizer state of a parameter the prune spared crosses the rebind (empty only before the first step)
test_eq(_probe.carried, [False] + [True] * 5)

In [ ]:
# agp prunes again and again across the fit, so the invariant is checked after each of those prunes
_probe = _OptProbe()
_learn = _probe_learner(agp, _probe)
_learn.fit(3)  # 3 epochs of 3 batches

test_eq(_probe.missed, [0] * 9)
assert len(set(_probe.channels)) > 2, f"agp should prune in several steps, got {_probe.channels}"
test_eq(_probe.channels[-1], 8)

# A state entry keyed by a parameter no group holds breaks `state_dict()`, so saving proves there is none
with tempfile.TemporaryDirectory() as _d:
    _learn.path = Path(_d)
    _learn.save('after_prune')

In [ ]:
# torch-pruning re-creates the parameters it replaces requiring grad — including those of a layer it only
# touches as a dependency of the one it prunes — so a frozen group has to be frozen again after the rebind
class _FrozenProbe(Callback):
    "Snapshot the frozen conv right after the prune that re-created its parameters"
    order = 60
    def __init__(self): self.snapshot = None
    def before_step(self):
        if self.snapshot is None and self.learn.model[0].out_channels < 16:
            self.snapshot = self.learn.model[3].weight.detach().clone()

# `freeze()` freezes every group but the last: conv '3' and its norm are the frozen ones here, and the
# prune reaches them through conv '0', which is trainable
def _frozen_first(m): return L(L(p for l in (m[3], m[4]) for p in l.parameters()),
                               L(p for l in (m[0], m[1], m[-1]) for p in l.parameters()))

_probe, _frozen = _OptProbe(), _FrozenProbe()
_learn = _probe_learner(one_shot, _probe, splitter=_frozen_first)
_learn.add_cb(_frozen)
_learn.freeze()
_head_bias = _learn.model[-1].bias.detach().clone()
_learn.fit(2)

test_eq(_probe.missed, [0] * 6)
test_eq(_learn.opt.frozen_idx, 1)
assert _frozen.snapshot is not None, "the model was never pruned"
# conv '3' was re-created by the prune and is frozen again: its weights are the ones the prune left
test_eq(_learn.model[3].weight.requires_grad, False)
assert torch.equal(_learn.model[3].weight.detach(), _frozen.snapshot), \
    "a frozen layer trained after the prune re-created its parameters"
# fastai's `train_bn` keeps a frozen group's norm layers training: that mark survives the rebind too
test_eq(_learn.model[4].weight.requires_grad, True)
# and the head, which was never frozen, did train
assert not torch.equal(_learn.model[-1].bias.detach(), _head_bias), "the head did not train"

In [ ]:
# A torch optimizer behind fastai's OptimWrapper keeps its groups and its state in the torch optimizer
class _MomentumProbe(Callback):
    "Record whether the conv that is being pruned holds a momentum buffer at each step"
    order = 60
    def __init__(self): self.fresh = []
    def before_step(self): self.fresh.append('exp_avg' in self.learn.opt.opt.state[self.learn.model[0].weight])

_probe, _mom = _OptProbe(), _MomentumProbe()
_learn = _probe_learner(agp, _probe, opt_func=partial(OptimWrapper, opt=torch.optim.Adam))
_learn.add_cb(_mom)
_learn.fit(2)

test_eq(_probe.missed, [0] * 6)
_live = {id(p) for g in _learn.opt.param_lists for p in g}
test_eq([n for n, p in _learn.model.named_parameters() if id(p) not in _live], [])
test_eq([k for k in _learn.opt.opt.state if id(k) not in _live], [])  # no key left on a dead parameter
# a parameter the prune spared keeps its momentum, one it replaced starts fresh
assert 'exp_avg' in _learn.opt.opt.state[_learn.model[-1].bias], "a spared parameter lost its momentum"
assert not _mom.fresh[0], "the conv the prune had just replaced carried a momentum buffer over"
assert _mom.fresh[-1], "the conv never accumulated momentum once the pruning had ended"

with tempfile.TemporaryDirectory() as _d:
    _learn.path = Path(_d)
    _learn.save('after_prune')  # torch's `state_dict()` raises KeyError on a state key outside the groups

In [ ]:
#| slow
# Full training with PruneCallback — verify parameter reduction
import warnings
from torch.utils.data import TensorDataset
from fastai.data.core import DataLoaders

_model = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
    nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 10)
)
_params_before = sum(p.numel() for p in _model.parameters())

_X = torch.randn(64, 3, 8, 8)
_y = torch.randint(0, 10, (64,))
_dls = DataLoaders.from_dsets(
    TensorDataset(_X[:48], _y[:48]),
    TensorDataset(_X[48:], _y[48:]),
    bs=16, device='cpu'
)

_cb = PruneCallback(pruning_ratio=0.3, schedule=one_shot, context='local', criteria=large_final)
_learn = Learner(_dls, _model, loss_func=nn.CrossEntropyLoss(), cbs=[_cb])
with warnings.catch_warnings(record=True) as _w:
    warnings.simplefilter("always")
    _learn.fit(3)

_params_after = sum(p.numel() for p in _model.parameters())
assert _params_after < _params_before, f"Expected params to decrease: {_params_before} → {_params_after}"

# The scheduled intermediates are fractions too — nothing looks like a percent during training
test_eq([x for x in _w if 'looks like a percent' in str(x.message)], [])
test_close(_model[0].out_channels / 16, 0.7, eps=0.05)

In [ ]:
#| slow
# Per-layer dict pruning — verify different layers reach different ratios
from torch.utils.data import TensorDataset
from fastai.data.core import DataLoaders

_model = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),   # '0','1','2'
    nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),  # '3','4','5'
    nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),  # '6','7','8'
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 10)         # '9','10','11'
)
_c0_before, _c3_before = _model[0].out_channels, _model[3].out_channels  # 16, 32
_params_before = sum(p.numel() for p in _model.parameters())

_X = torch.randn(64, 3, 8, 8)
_y = torch.randint(0, 10, (64,))
_dls = DataLoaders.from_dsets(
    TensorDataset(_X[:48], _y[:48]),
    TensorDataset(_X[48:], _y[48:]),
    bs=16, device='cpu'
)

# Prune conv '0' to 30% and conv '3' to 60% — neither feeds the (ignored) output Linear
_cb = PruneCallback(pruning_ratio={'0': 0.3, '3': 0.6}, schedule=one_shot, context='local', criteria=large_final)
_learn = Learner(_dls, _model, loss_func=nn.CrossEntropyLoss(), cbs=[_cb])
_learn.fit(3)

_params_after = sum(p.numel() for p in _model.parameters())
_c0_after, _c3_after = _model[0].out_channels, _model[3].out_channels

# Params dropped, and BOTH targeted layers shrank
assert _params_after < _params_before, f"Expected params to decrease: {_params_before} → {_params_after}"
assert _c0_after < _c0_before, f"conv '0' did not shrink: {_c0_before} → {_c0_after}"
assert _c3_after < _c3_before, f"conv '3' did not shrink: {_c3_before} → {_c3_after}"

# The 30%-target layer must retain a LARGER fraction of its channels than the 60%-target layer
_retained_0, _retained_3 = _c0_after / _c0_before, _c3_after / _c3_before
assert _retained_0 > _retained_3, (
    f"Per-layer targets not differentiated: conv '0' kept {_retained_0:.2f} (target 0.3), "
    f"conv '3' kept {_retained_3:.2f} (target 0.6)"
)
print(f"conv '0' (0.3 target): {_c0_before}→{_c0_after} ({_retained_0:.0%} kept) | "
      f"conv '3' (0.6 target): {_c3_before}→{_c3_after} ({_retained_3:.0%} kept)")